In [1]:
from promptsmith.dspy_init import get_dspy
dspy, lm = get_dspy()

/Users/yanivgal/dev/promptsmith/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## the dataset

In [2]:
import pandas as pd

df = pd.read_csv('../data/documents.csv')

## bulletizers

In [3]:
from promptsmith.tasks.bad_to_good.bulletize_text_1 import BulletizeText as simple_bulletizer
from promptsmith.tasks.bad_to_good.bulletize_text_2 import BulletizeText as better_bulletizer

simple_bulletizer = dspy.Predict(simple_bulletizer)
better_bulletizer = dspy.Predict(better_bulletizer)

## judges

In [4]:
from promptsmith.judges.judge_bullet_structure import JudgeBulletStructure
judge = dspy.Predict(JudgeBulletStructure)

## texts to bulletize

In [5]:
random_samples = df.sample(n=50, random_state=44)
results = []

for idx, row in random_samples.iterrows():
    text_to_bulletize = row['resource']

    b_simple = simple_bulletizer(input_text=text_to_bulletize)
    b_better = better_bulletizer(input_text=text_to_bulletize)

    r_simple = judge(text=b_simple.output_text)
    r_better = judge(text=b_better.output_text)

    result = {
        'text': text_to_bulletize,
        'simple_score': r_simple.score,
        'better_score': r_better.score,
        'simple_bullets': b_simple.output_text,
        'better_bullets': b_better.output_text,
        'simple_reasoning': r_simple.reasoning,
        'better_reasoning': r_better.reasoning
    }
    results.append(result)

results_df = pd.DataFrame(results)
print("Results summary:")
print(results_df[['simple_score', 'better_score']].describe())
print("\nDetailed results:")
print(results_df)

Results summary:
       simple_score  better_score
count     50.000000     50.000000
mean       0.355000      0.912000
std        0.101141      0.109991
min        0.250000      0.550000
25%        0.250000      0.850000
50%        0.350000      1.000000
75%        0.350000      1.000000
max        0.550000      1.000000

Detailed results:
                                                 text  simple_score  \
0   The following is the full transcript of an int...          0.35   
1   A man stood upon a railroad bridge in northern...          0.25   
2   ANT was first developed at the Centre de Socio...          0.35   
3   Key shared (commonly reported across groups) b...          0.55   
4   The error terrain analysis for machine learnin...          0.55   
5   Considering  NAT  as  a  complex  intervention...          0.25   
6   C-suite is continuously close to the business....          0.25   
7   Bilingual language control declines with age82...          0.25   
8   I mentioned at 

In [8]:
print(results_df.columns)

Index(['text', 'simple_score', 'better_score', 'simple_bullets',
       'better_bullets', 'simple_reasoning', 'better_reasoning'],
      dtype='object')


In [9]:
# Select only the desired columns
selected_columns = ['text', 'simple_bullets', 'better_bullets']
results_subset = results_df[selected_columns]

# Save to CSV
results_subset.to_csv('bulletized_results.csv', index=False)

In [10]:
results_df.to_csv('bulletized_results_full.csv', index=False)